# 3c - Read the PDF, with AI

**Time:** 45 minutes build + 20 minutes check  |  **AI: writes code from YOUR spec** - you explain every line

**What you will do**
- See what a PDF looks like to a machine.
- Write a spec: Goal, Inputs, Units and conventions, Output, Check.
- Ask Gemini to write a PDF reader from your spec. Paste it here.
- Explain every line in your own words.
- Then check the numbers: the parts must add up, two sources must agree, and a file of dates must be read the right way round.

The code is written by AI. The decision is yours: what you ask for decides what you get.

## The problem

The MD asks: **how much of Apple's profit really comes from Services?** The answer is in Apple's own statement of operations for the fourth quarter of fiscal 2025 (three months ended 27 September 2025), a PDF from apple.com.

You need four numbers: Products and Services, each for **net sales** and for **cost of sales**. Then the gross margin of each, and the share of Services in Apple's gross profit.

Reminder from Session 2: the spec decides the answer. This is the spec for the volatility line you read in 3a:

| | The example from 3a |
|---|---|
| **Goal** | Annual volatility of each stock |
| **Inputs** | `returns`, a DataFrame of daily returns |
| **Units and conventions** | decimals (0.28 = 28%), annual, 252 returns a year |
| **Output** | one number per stock |
| **Check** | Apple is about 28%; the index is below every single stock |

## Step 1 - Get the PDF

`pymupdf` is a library that reads PDFs. In Colab, one line installs it. Then we download the PDF from the course repo to the Colab folder.

In [ ]:
!pip install -q pymupdf

In [ ]:
import urllib.request

DATA = "https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/"
urllib.request.urlretrieve(DATA + "apple_q4_fy25_statements.pdf", "apple_q4_fy25_statements.pdf")
print("Saved: apple_q4_fy25_statements.pdf")

## Step 2 - What the machine sees

A human sees a table with columns. A machine gets **text, line by line**. `pymupdf.open(...)` opens the file, `doc[0]` is page 1 (positions start at 0), `.get_text()` is all the text of the page, and `.splitlines()` cuts it into a list of lines.

Below: 30 lines of page 1, starting just before the sales table, each between quotes with `repr`, so you can see **invisible spaces** and **empty lines**. Look at the lines after `'Net sales:'`.

**Your answer:** write down three things that will make a reader hard to write. For example: how many columns are there? Which line is Products? Is a number a number? Write here: ...

In [ ]:
import pymupdf

doc = pymupdf.open("apple_q4_fy25_statements.pdf")
raw = doc[0].get_text().splitlines()      # page 1, as a list of lines

print(len(raw), "lines on page 1")
for k, line in enumerate(raw[20:50], start=20):
    print(k, repr(line))

## Step 3 - Write your spec

Double-click this cell and fill it in. Be precise: AI fills every gap with its own guess. Think about what you just saw: which column? which units?

| | Your spec |
|---|---|
| **Goal** | ... |
| **Inputs** | ... |
| **Units and conventions** | ... |
| **Output** | ... |
| **Check** | ... |

## Step 4 - Ask Gemini, paste the code

1. Open Gemini (gemini.google.com, or the Gemini button in Colab).
2. Paste your spec, then ask: *"Write Python code with pymupdf for this spec."* The PDF is `apple_q4_fy25_statements.pdf`, in the same folder as this notebook.
3. Paste the code in the cell below.
4. Make the code leave the four numbers in variables called `products_sales`, `services_sales`, `products_cost` and `services_cost` (whole numbers), and print them. If Gemini gave other names, rename them.

Run the cell.

In [ ]:
# PASTE the code Gemini wrote here.
# The PDF is apple_q4_fy25_statements.pdf. Keep the variable names
# products_sales, services_sales, products_cost, services_cost.

## Step 5 - Explain every line

Rule of the course: you must be able to explain every line you submit. Copy the code above into the cell below as comments, or write next to each line what it does in your own words, in finance words.

Questions to answer for your own code: Where does it remove the invisible spaces? How does it turn `'73,716  $'` into a number? How does it find the right line: a position, or a label? What happens if Apple adds one line to the PDF next year?

Double-click here and write: ...

## Check - Do the numbers hold up?

AI wrote code, and it ran. That is not a check. Four checks. (Checks card, entries 9 to 12.)

First, a clean copy of the lines, written independently of the AI code (so the check does not depend on it): `pdf_lines`.

In [ ]:
pdf_lines = []
for line in pymupdf.open("apple_q4_fy25_statements.pdf")[0].get_text().splitlines():
    if line.strip() != "":
        pdf_lines.append(line.strip())

# Where the label "Total net sales (1)" is, in the clean list
k = pdf_lines.index("Total net sales (1)")
print(pdf_lines[k:k + 5])

### Check 1 - The parts add up

Products + Services must equal the total that the PDF prints itself. The line after `"Total net sales (1)"` is the quarter. Remove the commas and make it a number with `int(...)`.

In [ ]:
# YOUR CODE HERE: save the PDF's own total for the quarter in pdf_total (a whole number)

if products_sales + services_sales != pdf_total:
    raise ValueError(f"Parts do not add up: {products_sales} + {services_sales} is not {pdf_total}")
print(f"OK: {products_sales:,} + {services_sales:,} = {pdf_total:,}")

### Check 2 - The margins add up

The gross margin of Products plus the gross margin of Services must equal the line `"Gross margin"` of the PDF (the one after `"Total cost of sales"`). Same method: find the label with `.index(...)`, take the value on the next line.

In [ ]:
# YOUR CODE HERE: save the PDF's own gross margin for the quarter in pdf_gross_margin

my_gross_margin = (products_sales - products_cost) + (services_sales - services_cost)
if my_gross_margin != pdf_gross_margin:
    raise ValueError(f"Margins do not add up: {my_gross_margin} is not {pdf_gross_margin}")
print(f"OK: gross margin {my_gross_margin:,}")

### What the check catches

A common AI mistake: the wrong column (last year, or the twelve months), or the wrong "Products" line (cost of sales instead of net sales). The sum check catches both. Run the cell: it takes last year's Services (the second value) on purpose and runs the same check. It prints the message instead of stopping.

In [ ]:
wrong_services = int(pdf_lines[pdf_lines.index("Services") + 2].replace(",", ""))   # last year's column

try:
    if products_sales + wrong_services != pdf_total:
        raise ValueError(f"Parts do not add up: {products_sales} + {wrong_services} is not {pdf_total}")
except ValueError as error:
    print("The check caught it:", error)

### Check 3 - Two sources must agree

The PDF also holds the twelve-month total (the third value after `"Total net sales (1)"`). Apple's fiscal 2025 revenue in the SEC data (the source of the Session 0 demo) is **416,161** million dollars. The two sources must agree.

In [ ]:
sec_revenue_fy2025 = 416161     # Apple FY2025 revenue, SEC company facts, USD millions

# YOUR CODE HERE: save the PDF's twelve-month total for 2025 in pdf_total_12m (a whole number)

if pdf_total_12m != sec_revenue_fy2025:
    raise ValueError(f"The PDF says {pdf_total_12m}, the SEC says {sec_revenue_fy2025}")
print(f"OK: PDF and SEC agree on {sec_revenue_fy2025:,}")

### Check 4 - The dates

A colleague in Paris sends `prices_monthly_fr.csv`: monthly Apple prices, one per month, dated the first of the month in French format `dd/mm/yyyy`. Look at the raw file first.

In [ ]:
import pandas as pd

raw_text = pd.read_csv(DATA + "prices_monthly_fr.csv", dtype=str)   # read as plain text
print(raw_text.head(3))
print("...")
print(raw_text.tail(1))
print(len(raw_text), "rows")

Now read it the usual way, with `parse_dates=["Date"]`, and ask: how many months are in the second half of 2024? Twelve months of data should give six.

In [ ]:
fr = pd.read_csv(DATA + "prices_monthly_fr.csv", parse_dates=["Date"])

print(fr["Date"].head(3))
print("Rows in the second half of 2024:", len(fr[fr["Date"] >= "2024-07-01"]))
print("Different months in the file:", fr["Date"].dt.month.nunique())

**Your answer:** no error, but 0 rows. What did pandas do with `01/02/2024`? Which date did it read? Why is the bug silent? Write 2 or 3 lines here: ...

The fix: tell pandas the day comes first, with `dayfirst=True`. Then check **the first date, the last date and the count** (checks card, entry 9), and that the dates are oldest first.

In [ ]:
# YOUR CODE HERE: read the file again, with dayfirst=True, and save it in fr_ok

first_date = fr_ok["Date"].min()
last_date = fr_ok["Date"].max()
count = len(fr_ok)

print("First date:", first_date.date())
print("Last date: ", last_date.date())
print("Count:     ", count)

if fr_ok["Date"].dt.month.nunique() != count:
    raise ValueError("Twelve rows should be twelve different months.")
if not fr_ok["Date"].is_monotonic_increasing:
    raise ValueError("The dates are not oldest first.")
print("OK: one row per month, oldest first")

**Your answer:** compare with the raw file above. Is the first date the first line of the file, read the way a Parisian reads it? Is the last date right: `01/12/2024` is 1 December or 12 January? How many rows are now in the second half of 2024? Write 2 lines here: ...

## The answer for the MD

Gross margin is net sales minus cost of sales. The gross margin **percentage** divides it by the net sales.

You should get gross margin **36.2%** for Products and **75.3%** for Services. Services are **28.1%** of net sales, but **44.8%** of gross profit.

In [ ]:
# YOUR CODE HERE: save these four numbers (decimals):
# products_margin   gross margin of Products, as a share of its net sales
# services_margin   gross margin of Services, as a share of its net sales
# services_share_sales    Services net sales / total net sales
# services_share_profit   Services gross margin / total gross margin

print(f"Gross margin, Products: {products_margin:.1%}")
print(f"Gross margin, Services: {services_margin:.1%}")
print(f"Services share of net sales:    {services_share_sales:.1%}")
print(f"Services share of gross profit: {services_share_profit:.1%}")

**Your answer for the MD:** write two sentences. How much of Apple's profit comes from Services? Why is the share of profit so different from the share of sales? Write here: ...

## What you learned

**Python**
- **Library + import**: `import pymupdf`, `pymupdf.open("file.pdf")`, `doc[0]` is page 1 (from 0), `.get_text()` its text.
- **Text methods**: `.splitlines()` cuts text into a list of lines, `.strip()` removes invisible spaces, `.replace(",", "")` removes the commas.
- **Convert**: `int("28750")` turns text into a whole number.
- **Find in a list**: `lines.index("Services")` is the position of the first match. `lines[i + 1]` is the next line.
- **Loop + condition + append** to clean the lines, as in Session 1.
- **Dates**: `pd.read_csv(..., parse_dates=["Date"], dayfirst=True)`. **Guard** with `raise` to stop the code when a check fails.

**Finance**
- A PDF is text, not a table: one value per line, four columns, "Products" twice. A reader must say which column and which units.
- Services are 28.1% of Apple's net sales but 44.8% of its gross profit (gross margin 75.3% against 36.2% for Products).
- Checks 9 to 12: check the dates (first, last, count, oldest first); know the source's units and conventions; the parts must add up to the total; two sources must agree.
- Day-first dates read month-first give no error and no answer: check the first date, the last date and the count.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right